# Elite Track t-SNE Grid

For each iteration subfolder under `data/ns/images/`, load `elite_images.npz`,
compute **fully rotation-invariant** descriptors via the 2-D Fourier power spectrum
radial profile, run t-SNE, and render a grid where every point is the actual track image.

**Why the radial FFT profile is fully rotation-invariant:**  
A rotation of the image rotates its 2-D DFT by the same angle. Taking the
power spectrum (squared magnitude) and then averaging over all angles at each
radius yields a 1-D profile that depends only on radial frequency — not on
orientation. No alignment or canonical-pose step is needed.

In [ ]:
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — avoids kernel crash when no display is available
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
import os
import glob

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
IMAGES_ROOT   = 'data/ns/images'  # root that contains per-iteration subfolders
OUTPUT_DIR    = os.path.dirname(os.path.abspath(
    globals().get('__vsc_ipynb_file__', 'qd/visualizations/elites_tsne.ipynb')
))
PLOTS_DIR     = os.path.join(OUTPUT_DIR, 'plots')
N_RADIAL_BINS = 64    # length of the descriptor vector
TSNE_ITERS    = 1000
MIN_ELITES    = 5     # skip iterations with fewer elites than this

# NPZ keys produced by ArchiveVisualizer.save_elite_images:
#   'plain'            – white-bg crimson-line images used for t-SNE descriptors
#   'colored_<metric>' – per-metric colored images (one set per COLORING_METRICS entry)

In [5]:
def rotation_invariant_descriptor(img_rgba: np.ndarray, n_bins: int = N_RADIAL_BINS) -> np.ndarray:
    """Return a 1-D rotation-invariant descriptor from an RGBA track image.

    Steps
    -----
    1. Convert to grayscale.
    2. Compute the 2-D DFT and take the power spectrum.
    3. Bin pixels by radial distance from the DC component (centre of the
       shifted spectrum) and average power within each ring.
    4. Log-compress and normalise to [0, 1].

    A rotation of the image rotates the 2-D DFT by the same angle; averaging
    over the angular dimension makes the profile completely rotation-invariant.
    """
    gray  = img_rgba[:, :, :3].mean(axis=2).astype(np.float32)
    fft2  = np.fft.fft2(gray)
    power = np.abs(np.fft.fftshift(fft2)) ** 2          # (H, W)

    H, W  = power.shape
    cy, cx = H / 2.0, W / 2.0
    y_idx, x_idx = np.indices((H, W))
    r = np.sqrt((x_idx - cx) ** 2 + (y_idx - cy) ** 2)  # float radii

    max_r   = min(cy, cx)
    r_bin   = np.clip((r / max_r * n_bins).astype(int), 0, n_bins - 1)

    # Fast radial average with bincount
    flat_bins   = r_bin.ravel()
    flat_power  = power.ravel()
    sums   = np.bincount(flat_bins, weights=flat_power, minlength=n_bins)
    counts = np.bincount(flat_bins,                     minlength=n_bins)
    profile = np.where(counts > 0, sums / counts, 0.0)

    profile = np.log1p(profile)      # log-compress to reduce dynamic range
    if profile.max() > 0:
        profile /= profile.max()
    return profile

In [6]:
npz_paths = sorted(glob.glob(os.path.join(IMAGES_ROOT, '*', 'elite_images.npz')))
print(f'Found {len(npz_paths)} iteration folder(s)')
for p in npz_paths:
    print(' ', p)

Found 2 iteration folder(s)
  data/ns/images\15\elite_images.npz
  data/ns/images\30\elite_images.npz


In [ ]:
os.makedirs(PLOTS_DIR, exist_ok=True)

for npz_path in npz_paths:
    folder_name = os.path.basename(os.path.dirname(npz_path))

    data         = np.load(npz_path)
    plain_images = data['plain']   # (N, H, W, 4) uint8 RGBA — used for t-SNE
    ids          = data['ids']     # (N,)
    N            = len(plain_images)

    if N < MIN_ELITES:
        print(f'Skipping iteration {folder_name}: only {N} elites (< {MIN_ELITES})')
        continue

    # Collect all image variants: plain + any colored_* present in the NPZ
    image_variants = {'plain': plain_images}
    for key in data.files:
        if key.startswith('colored_'):
            image_variants[key] = data[key]

    print(f'\nIteration {folder_name} — {N} elites, variants: {list(image_variants.keys())}')

    # ── Descriptors on plain images only ────────────────────────────────────
    descriptors = np.stack([rotation_invariant_descriptor(img) for img in plain_images])
    descriptors = StandardScaler().fit_transform(descriptors)

    # ── t-SNE computed once, on plain descriptors ────────────────────────────
    perplexity = min(30, max(2, N // 3))
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        max_iter=TSNE_ITERS,
        random_state=42,
        init='pca',
        learning_rate='auto',
    )
    coords = tsne.fit_transform(descriptors)   # (N, 2)
    print(f'  t-SNE done  perplexity={perplexity}  KL={tsne.kl_divergence_:.3f}')

    # ── Assign t-SNE points to grid cells (no overlaps) ──────────────────────
    lo, hi      = coords.min(axis=0), coords.max(axis=0)
    coords_norm = (coords - lo) / (hi - lo + 1e-8)

    cols     = int(np.ceil(np.sqrt(N)))
    rows     = int(np.ceil(N / cols))
    gx       = np.linspace(0, 1, cols)
    gy       = np.linspace(0, 1, rows)
    gxx, gyy = np.meshgrid(gx, gy)
    grid_pos = np.stack([gxx.ravel(), gyy.ravel()], axis=1)   # (rows*cols, 2)

    cost = cdist(coords_norm, grid_pos)
    _, col_ind = linear_sum_assignment(cost)   # col_ind[i] = grid cell for image i

    # ── Grid plots: one per image variant ────────────────────────────────────
    for variant_name, images in image_variants.items():
        cell_px = 1.5
        fig, axes = plt.subplots(rows, cols,
                                 figsize=(cols * cell_px, rows * cell_px),
                                 facecolor='white')
        axes = np.asarray(axes).ravel()

        for ax in axes:
            ax.set_facecolor('white')
            ax.set_axis_off()

        for img_idx, cell_idx in enumerate(col_ind):
            axes[cell_idx].imshow(images[img_idx][:, :, :3], interpolation='lanczos')
            axes[cell_idx].set_axis_off()

        fig.suptitle(
            f'Elite tracks — iteration {folder_name}  [{variant_name}]  (N={N})',
            color='black', fontsize=11, y=1.01,
        )
        plt.subplots_adjust(wspace=0.02, hspace=0.02)

        out_path = os.path.join(PLOTS_DIR, f'tsne_grid_{folder_name}_{variant_name}.png')
        fig.savefig(out_path, dpi=120, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        print(f'  Saved → {out_path}')

    # ── Scatter plots: images placed at raw t-SNE coords, one per metric ──────
    zoom   = min(0.05, 1.0 / np.sqrt(N))   # shrink thumbnails for large archives
    margin = (coords.max(axis=0) - coords.min(axis=0)).max() * 0.08

    for key in data.files:
        if not key.startswith('colored_'):
            continue
        label  = key[len('colored_'):]   # e.g. 'curvature_entropy'
        images = data[key]               # (N, H, W, 4) uint8 RGBA

        fig, ax = plt.subplots(figsize=(10, 10), facecolor='white')
        ax.set_facecolor('white')

        for i, (x, y) in enumerate(coords):
            im = OffsetImage(images[i][:, :, :3], zoom=zoom, interpolation='lanczos')
            ab = AnnotationBbox(im, (x, y), frameon=False, pad=0)
            ax.add_artist(ab)

        ax.set_xlim(coords[:, 0].min() - margin, coords[:, 0].max() + margin)
        ax.set_ylim(coords[:, 1].min() - margin, coords[:, 1].max() + margin)
        ax.set_title(
            f't-SNE — iteration {folder_name}  [{label}]  (N={N})',
            fontsize=11,
        )
        ax.set_axis_off()
        plt.tight_layout()

        out_path = os.path.join(PLOTS_DIR, f'tsne_scatter_{folder_name}_{label}.png')
        fig.savefig(out_path, dpi=120, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        print(f'  Saved → {out_path}')